# Find-Enemy Training (Static Enemy, Single Police)

Goal: **train one police agent to find and go near a static enemy dot**, using your own drawn maps with walls.

Workflow:
- **Step 1 (local)**: Use the editor (e.g. `play_model.py`) to draw a map, place police (blue) and enemy (red), and save as `training_map.json`.
- **Step 2 (Colab or local notebook)**: Upload/copy `training_map.json` into this project folder and run the training cell; the env loads your map and trains the police.
- **Step 3**: Repeat with more advanced maps and continue training to improve the policy.

The enemy does **not move** in this notebook; only the police is trained to navigate toward the enemy.


In [ ]:
# If running in Colab: clone repo and install dependencies (run once per session)
import sys, os
if "google.colab" in sys.modules:
    %cd /content
    if not os.path.exists("pursuit_arena"):
        !git clone https://github.com/GraslyDias/pursuit_arena.git
    %cd /content/pursuit_arena
    !pip install pygame gymnasium stable-baselines3[extra] numpy


## Upload or provide `training_map.json`

If you're running in Colab, upload your locally saved `training_map.json` from the editor.


In [ ]:
import sys
if "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()  # choose training_map.json from your computer


## Train POLICE to find static enemy on your map

This uses `ChaseEscapeEnv` with `static_enemy=True` and **always** loads `training_map.json` if present.


In [ ]:
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

import pursuit_arena.ai.rl.chase_escape_env as ce
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

# Reward config for FIND-ENEMY task (5-second budget, strong signal for finding enemy)
# You can tune these numbers.
ce.POLICE_REWARD_ARREST = 10.0        # big reward for reaching (arresting) the enemy
ce.POLICE_REWARD_ESCAPE = 0.0         # enemy never moves; escape is not focus here
ce.POLICE_REWARD_VISIBLE = 0.2        # reward per step when enemy is in FOV/LOS
ce.POLICE_REWARD_APPROACH = 0.05      # bonus when police moves closer to enemy (good path)
ce.POLICE_PENALTY_RETREAT = -0.02     # penalty when police moves farther from enemy (bad path)
ce.POLICE_PENALTY_WALL_COLLISION = -0.2  # stronger negative when bumping walls
ce.POLICE_STEP_PENALTY = -0.001
ce.POLICE_TIMEOUT_PENALTY = -5.0      # strong negative when time is over and enemy not found

log_dir = Path("runs/ppo_find_enemy")
log_dir.mkdir(parents=True, exist_ok=True)

MAX_STEPS_PER_EPISODE = 300  # ~5 seconds if you imagine 60 FPS

def make_env():
    env = ChaseEscapeEnv(render_mode=None, max_steps=MAX_STEPS_PER_EPISODE)
    env.static_enemy = True  # enemy does not move
    map_path = Path("training_map.json")
    if map_path.exists():
        env.training_map_options = load_training_map(map_path)
        print("Using training_map.json for find-enemy training")
    env = Monitor(env, str(log_dir / "monitor_find_enemy.csv"))
    return env

vec_env = DummyVecEnv([make_env])

model_path = log_dir / "ppo_find_enemy_final.zip"
if model_path.exists():
    model = PPO.load(str(model_path), env=vec_env, device="cpu")
    print("Loaded existing find-enemy model, continuing training.")
else:
    model = PPO("MlpPolicy", vec_env,
                verbose=1,
                tensorboard_log=str(log_dir / "tb_find_enemy"),
                device="cpu")
    print("Training find-enemy model from scratch.")

TOTAL_TIMESTEPS = 200_000  # adjust as you like
model.learn(total_timesteps=TOTAL_TIMESTEPS)
model.save(str(model_path))
print("Saved find-enemy model to", model_path)
vec_env.close()

# --- Simple evaluation to show episode rewards and steps ---
env_eval = ChaseEscapeEnv(render_mode=None, max_steps=MAX_STEPS_PER_EPISODE)
env_eval.static_enemy = True
map_path = Path("training_map.json")
if map_path.exists():
    env_eval.training_map_options = load_training_map(map_path)

model_eval = PPO.load(str(model_path), env=env_eval, device="cpu")

num_episodes = 10
for ep in range(num_episodes):
    obs, info = env_eval.reset()
    done, truncated = False, False
    ep_reward = 0.0
    steps = 0
    while not (done or truncated):
        action, _ = model_eval.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env_eval.step(action)
        ep_reward += float(reward)
        steps += 1
    print(f"Episode {ep+1}: steps={steps}, reward={ep_reward:.3f}, info={info}")

env_eval.close()


## Quick visual check (optional, local Jupyter)

If you're running this notebook **locally** (not Colab), you can use `render_mode='human'` to open a window and watch the agent.


In [ ]:
from stable_baselines3 import PPO
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

try:
    # Only works on local machine with a display (not in Colab)
    env = ChaseEscapeEnv(render_mode="human")
    env.static_enemy = True
    map_path = Path("training_map.json")
    if map_path.exists():
        env.training_map_options = load_training_map(map_path)
    model = PPO.load("runs/ppo_find_enemy/ppo_find_enemy_final.zip", env=env)

    episodes = 3
    for ep in range(episodes):
        obs, info = env.reset()
        done = False
        truncated = False
        while not (done or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(action)
    env.close()
except Exception as e:
    print("Visual run skipped or failed (likely due to no display):", e)


## Upload a new map and continue training

Use this section when you draw a **new map** (walls + police/enemy) in the local editor and save it as `training_map.json`. You can upload the new file and continue training the same find-enemy model on this new layout.

In [ ]:
import sys
if "google.colab" in sys.modules:
    from google.colab import files
    print("Upload NEW training_map.json for continued training...")
    uploaded = files.upload()  # choose the updated training_map.json
else:
    print("Not running in Colab – make sure training_map.json is updated locally.")

In [ ]:
# Continue training find-enemy model on the (possibly new) training_map.json
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

import pursuit_arena.ai.rl.chase_escape_env as ce
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

log_dir = Path("runs/ppo_find_enemy")
log_dir.mkdir(parents=True, exist_ok=True)

# You can change this to control episode length and extra training steps
MAX_STEPS_PER_EPISODE = 300   # steps per episode
MORE_TIMESTEPS = 100_000      # additional timesteps to train on the new map

assert (log_dir / "ppo_find_enemy_final.zip").exists(), "Base find-enemy model not found. Run the main training cell first."


def make_env_new():
    env = ChaseEscapeEnv(render_mode=None, max_steps=MAX_STEPS_PER_EPISODE)
    env.static_enemy = True
    map_path = Path("training_map.json")
    assert map_path.exists(), "training_map.json not found. Upload or copy it first."
    env.training_map_options = load_training_map(map_path)
    print("Continuing training on training_map.json for find-enemy task")
    env = Monitor(env, str(log_dir / "monitor_find_enemy_new.csv"))
    return env


vec_env_new = DummyVecEnv([make_env_new])
model_path = log_dir / "ppo_find_enemy_final.zip"
model_new = PPO.load(str(model_path), env=vec_env_new, device="cpu")
print("Loaded existing find-enemy model. Training more on the new map...")

model_new.learn(total_timesteps=MORE_TIMESTEPS)
model_new.save(str(model_path))
print("Saved updated find-enemy model to", model_path)
vec_env_new.close()

## Train with random enemy position (map without enemy)

Use this when your `training_map.json` has **only walls (and maybe police)** but **no enemy**.

- The environment will load your walls from `training_map.json`.
- Each episode, it will **randomly place the enemy** at a different position.
- The police is still trained to find a **static enemy dot**, but the spawn is random for more general training.
- This cell **continues training the existing find-enemy model** (`ppo_find_enemy_final.zip`).

In [ ]:
# Continue training: random enemy position on your wall map (no fixed enemy in JSON)
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

import pursuit_arena.ai.rl.chase_escape_env as ce
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

log_dir = Path("runs/ppo_find_enemy")
log_dir.mkdir(parents=True, exist_ok=True)

# Episode length and extra timesteps for this random-enemy training
MAX_STEPS_PER_EPISODE = 300   # you can change this
MORE_TIMESTEPS_RANDOM = 100_000  # you can change this

model_path = log_dir / "ppo_find_enemy_final.zip"
assert model_path.exists(), "Base find-enemy model not found. Run the main training cell first."


def make_env_random_enemy():
    """Env that loads walls from training_map.json but randomizes enemy spawn each episode."""
    env = ChaseEscapeEnv(render_mode=None, max_steps=MAX_STEPS_PER_EPISODE)
    env.static_enemy = True  # enemy does not move once spawned

    map_path = Path("training_map.json")
    assert map_path.exists(), "training_map.json not found. Upload or copy it first."
    # Load walls (and possibly police start) from map; ignore enemy from file.
    options = load_training_map(map_path)
    # If the JSON has an enemy, we drop it so that env uses its own random spawn.
    options["enemies"] = []
    env.training_map_options = options

    env = Monitor(env, str(log_dir / "monitor_find_enemy_random_enemy.csv"))
    return env


vec_env_random = DummyVecEnv([make_env_random_enemy])
model_random = PPO.load(str(model_path), env=vec_env_random, device="cpu")
print("Loaded existing find-enemy model. Training more with random enemy positions...")

model_random.learn(total_timesteps=MORE_TIMESTEPS_RANDOM)
model_random.save(str(model_path))
print("Saved updated find-enemy model to", model_path)
vec_env_random.close()

## Train with fully random walls + random police/enemy positions

This section trains the **find-enemy** policy on:

- **Random police start position** every episode.
- **Random enemy position** every episode (enemy is still static once spawned).
- **Random wall layouts** generated by the environment (sometimes few walls, sometimes more, different shapes).

You can control:
- Whether to **start from your current model** or **train a brand new model**.
- Whether to **use your uploaded `training_map.json` walls** or rely purely on **random maps**.

Useful when you want a very **general finder** that works on many layouts, not only the exact maps you drew.

In [ ]:
# General find-enemy training: random walls + random police/enemy positions
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

import pursuit_arena.ai.rl.chase_escape_env as ce
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

# ==== TOGGLES ====
# If True: continue from existing ppo_find_enemy_final.zip (if it exists).
# If False: always start a NEW model from scratch.
USE_EXISTING_MODEL = True

# If True: use walls from training_map.json (if present).
# If False: let the environment generate fully random walls each episode.
USE_TRAINING_MAP = False

# Episode length and total timesteps for this general training run
MAX_STEPS_PER_EPISODE = 300       # steps per episode (you can change)
TOTAL_TIMESTEPS_GENERAL = 200_000  # total timesteps to train (you can change)


# (Optional) adjust reward shaping again here if you want
ce.POLICE_REWARD_ARREST = 15.0
ce.POLICE_REWARD_ESCAPE = 0.0
ce.POLICE_REWARD_VISIBLE = 2
ce.POLICE_REWARD_APPROACH = 3      # reward for moving closer to enemy without hitting walls
ce.POLICE_PENALTY_RETREAT = -2     # penalty for moving away from enemy
ce.POLICE_PENALTY_WALL_COLLISION = -3
ce.POLICE_STEP_PENALTY = -0.001
ce.POLICE_TIMEOUT_PENALTY = -5.0


log_dir = Path("runs/ppo_find_enemy")
log_dir.mkdir(parents=True, exist_ok=True)
model_path = log_dir / "ppo_find_enemy_final.zip"


def make_env_general():
    """Env with random police/enemy + random walls, optionally using walls from training_map.json."""
    env = ChaseEscapeEnv(render_mode=None, max_steps=MAX_STEPS_PER_EPISODE)
    env.static_enemy = True  # enemy stays as a static dot once spawned

    map_path = Path("training_map.json")
    if USE_TRAINING_MAP and map_path.exists():
        # Load walls (and possibly police/enemy) but drop fixed positions so env randomizes them.
        options = load_training_map(map_path)
        options["police"] = []   # force random police position
        options["enemies"] = []  # force random enemy position
        env.training_map_options = options
        print("Using walls from training_map.json, but random police/enemy positions.")
    else:
        # No training_map_options: env will generate fully random walls + random spawns.
        print("Using fully random walls and random police/enemy positions.")

    env = Monitor(env, str(log_dir / "monitor_find_enemy_general.csv"))
    return env


vec_env_general = DummyVecEnv([make_env_general])

if USE_EXISTING_MODEL and model_path.exists():
    model_general = PPO.load(str(model_path), env=vec_env_general, device="cpu")
    print("Loaded existing find-enemy model. Continuing training on general random layouts...")
else:
    model_general = PPO(
        "MlpPolicy",
        vec_env_general,
        verbose=1,
        tensorboard_log=str(log_dir / "tb_find_enemy_general"),
        device="cpu",
    )
    if USE_EXISTING_MODEL and not model_path.exists():
        print("No existing model found, training a NEW general find-enemy model from scratch.")
    else:
        print("Training a NEW general find-enemy model from scratch (USE_EXISTING_MODEL = False).")


model_general.learn(total_timesteps=TOTAL_TIMESTEPS_GENERAL)
model_general.save(str(model_path))
print("Saved general find-enemy model to", model_path)
vec_env_general.close()